1. Load the libraries

In [1]:
import os
import warnings
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
import numpy as np

warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

2. Define the model

In [2]:
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])

3. Define loss function and optimizer

In [3]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

4. Implement custom training loop

In [4]:
epochs = 2
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)
            loss_value = loss_fn(y_batch_train, logits)

        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')

Start of epoch 1
Epoch 1 Step 0: Loss = 2.2299985885620117
Epoch 1 Step 200: Loss = 0.4286557137966156
Epoch 1 Step 400: Loss = 0.18054255843162537
Epoch 1 Step 600: Loss = 0.1761917620897293
Epoch 1 Step 800: Loss = 0.15782828629016876
Epoch 1 Step 1000: Loss = 0.4148601293563843
Epoch 1 Step 1200: Loss = 0.1755494326353073
Epoch 1 Step 1400: Loss = 0.2876662611961365
Epoch 1 Step 1600: Loss = 0.2004455029964447
Epoch 1 Step 1800: Loss = 0.17619509994983673
Start of epoch 2
Epoch 2 Step 0: Loss = 0.09032069146633148
Epoch 2 Step 200: Loss = 0.17322532832622528
Epoch 2 Step 400: Loss = 0.0979585275053978
Epoch 2 Step 600: Loss = 0.04739409312605858
Epoch 2 Step 800: Loss = 0.10459943860769272
Epoch 2 Step 1000: Loss = 0.22143873572349548
Epoch 2 Step 1200: Loss = 0.09188611805438995
Epoch 2 Step 1400: Loss = 0.14776647090911865
Epoch 2 Step 1600: Loss = 0.16819429397583008
Epoch 2 Step 1800: Loss = 0.10695558786392212


5. Add accuracy metrics

In [5]:
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()
epochs = 5
for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)
            loss_value = loss_fn(y_batch_train, logits)

        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        accuracy_metric.update_state(y_batch_train, logits)

        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')

    accuracy_metric.reset_state()

Start of epoch 1
Epoch 1 Step 0: Loss = 0.04212632402777672 Accuracy = 1.0
Epoch 1 Step 200: Loss = 0.11877071857452393 Accuracy = 0.9762126803398132
Epoch 1 Step 400: Loss = 0.06636881828308105 Accuracy = 0.973192036151886
Epoch 1 Step 600: Loss = 0.022329552099108696 Accuracy = 0.9738456606864929
Epoch 1 Step 800: Loss = 0.06460396200418472 Accuracy = 0.9740558862686157
Epoch 1 Step 1000: Loss = 0.11049624532461166 Accuracy = 0.9741820693016052
Epoch 1 Step 1200: Loss = 0.049990274012088776 Accuracy = 0.9747085571289062
Epoch 1 Step 1400: Loss = 0.07629698514938354 Accuracy = 0.9749955534934998
Epoch 1 Step 1600: Loss = 0.08936597406864166 Accuracy = 0.9748399257659912
Epoch 1 Step 1800: Loss = 0.0673666000366211 Accuracy = 0.9753435850143433
Start of epoch 2
Epoch 2 Step 0: Loss = 0.02006290666759014 Accuracy = 1.0
Epoch 2 Step 200: Loss = 0.09633708745241165 Accuracy = 0.9838308691978455
Epoch 2 Step 400: Loss = 0.047430865466594696 Accuracy = 0.9816084504127502
Epoch 2 Step 600: L

6. Custom callback for advanced logging

In [6]:
from tensorflow.keras.callbacks import Callback
class CustomCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}')

epochs = 2
custom_callback = CustomCallback()

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:

            logits = model(x_batch_train, training=True)

            loss_value = loss_fn(y_batch_train, logits)


        grads = tape.gradient(loss_value, model.trainable_weights)

        optimizer.apply_gradients(zip(grads, model.trainable_weights))


        accuracy_metric.update_state(y_batch_train, logits)


        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')

    custom_callback.on_epoch_end(epoch, logs={'loss': loss_value.numpy(), 'accuracy': accuracy_metric.result().numpy()})

    accuracy_metric.reset_state()

Start of epoch 1
Epoch 1 Step 0: Loss = 0.02188401110470295 Accuracy = 1.0
Epoch 1 Step 200: Loss = 0.010423355735838413 Accuracy = 0.9947139024734497
Epoch 1 Step 400: Loss = 0.01947048492729664 Accuracy = 0.9949345588684082
Epoch 1 Step 600: Loss = 0.03533385321497917 Accuracy = 0.99558025598526
Epoch 1 Step 800: Loss = 0.022548070177435875 Accuracy = 0.9948501586914062
Epoch 1 Step 1000: Loss = 0.06282714009284973 Accuracy = 0.9950361847877502
Epoch 1 Step 1200: Loss = 0.02117585949599743 Accuracy = 0.9952643513679504
Epoch 1 Step 1400: Loss = 0.030582081526517868 Accuracy = 0.9949589371681213
Epoch 1 Step 1600: Loss = 0.007178670261055231 Accuracy = 0.9949055314064026
Epoch 1 Step 1800: Loss = 0.0172550268471241 Accuracy = 0.9948639869689941
End of epoch 1, loss: 0.003748243907466531, accuracy: 0.9949833154678345
Start of epoch 2
Epoch 2 Step 0: Loss = 0.018963268026709557 Accuracy = 1.0
Epoch 2 Step 200: Loss = 0.03965603560209274 Accuracy = 0.9956467747688293
Epoch 2 Step 400: Lo

7. Add hidden layers

In [7]:
input_layer = Input(shape=(28, 28))
hidden_layer1 = Dense(64, activation='relu')(input_layer)
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)
output_layer = Dense(1, activation='sigmoid')(hidden_layer2)
model = Model(inputs=input_layer, outputs=output_layer)


8. Compile the model

In [8]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

9. Train the model

In [9]:
model = Sequential([
    Input(shape=(20,)),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

X_train = np.random.rand(1000, 20)
y_train = np.random.randint(2, size=(1000, 1))

model.fit(X_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.4882 - loss: 0.7023 
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5268 - loss: 0.6931 
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5476 - loss: 0.6869 
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5974 - loss: 0.6810 
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5442 - loss: 0.6827 
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5648 - loss: 0.6791 
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5385 - loss: 0.6829 
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6171 - loss: 0.6728 
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6072 - loss: 0.6690 
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5494 - loss: 0.6789 


10. Evaluate the model

In [10]:
X_test = np.random.rand(200, 20)
y_test = np.random.randint(2, size=(200, 1))
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test loss: {loss}')
print(f'Test accuracy: {accuracy}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5521 - loss: 0.6871  
Test loss: 0.6753407120704651
Test accuracy: 0.6000000238418579


Hyperparameter tuning

In [11]:
!pip install keras-tuner

  Obtaining dependency information for keras-tuner from https://files.pythonhosted.org/packages/db/5d/945296512980b0827e93418514c8be9236baa6f0a1e8ca8be3a2026665b0/keras_tuner-1.4.7-py3-none-any.whl.metadata
  Obtaining dependency information for kt-legacy from https://files.pythonhosted.org/packages/16/53/aca9f36da2516db008017db85a1f3cafaee0efc5fc7a25d94c909651792f/kt_legacy-1.0.5-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/129.1 kB ? eta -:--:--
   --- ------------------------------------ 10.2/129.1 kB ? eta -:--:--
   ---------------------------- ----------- 92.2/129.1 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 129.1/129.1 kB 1.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import json
import os
import keras_tuner as kt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=20, n_classes=2)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int('units', min_value=32, max_value=512, step=32),
                    activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(hp.Float('learning_rate', 1e-4, 1e-2, sampling='LOG')),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='tuner_results',
    project_name='hyperparam_tuning'
)

tuner.search(X_train, y_train, validation_data=(X_val, y_val), epochs=5)

try:
    for i in range(10):

        best_hps = tuner.get_best_hyperparameters(num_trials=10)[0]

        results = {
            "trial": i + 1,
            "hyperparameters": best_hps.values,
            "score": None
        }
        os.makedirs('tuning_results', exist_ok=True)
        with open(os.path.join('tuning_results', f"trial_{i + 1}.json"), "w") as f:
            json.dump(results, f)

except IndexError:
    print("Tuning process has not completed or no results available.")

Trial 10 Complete [00h 00m 02s]
val_accuracy: 0.8799999952316284

Best val_accuracy So Far: 0.9049999713897705
Total elapsed time: 00h 00m 20s
